# Перебор признаков: заказы и позиции

In [ ]:
'''
Что известно:
- 222 признака, LightGBM + XGBoost + CatBoost с подбором: 0.726 на кросс-валидации, около 0.72 на Public;
- по словам преподавателя, решение уровня Middle (0.837) построено только на признаках, и даже простые модели
  набирают этот скор; значит, нам не хватает одного-двух сильных признаков;
- у клиентов с 2–3 днями заказов target почти предопределён днём «последний заказ + средний интервал»;
  для 4+ дней такой формулы мы не нашли, хотя перебирали её вручную и только в днях;
- метки train ведут себя не так, как покупки, видимые в самой истории.

Что показал перебор 684 кандидатов (ноутбук _feature_search):
- волшебного одиночного признака нет: лучший даёт AP@20% 0.52 против 0.50 у формулы;
- но признаки по отдельным заказам (заказы в один день не склеиваются) дают LightGBM +0.014,
  0.718 -> 0.732, плюс во всех 9 парах;
- ни одно естественное определение target не воспроизводит train.

Что делаем здесь — копаем глубже в сторону заказов и позиций:
- опорный набор: 222 признака + признаки по отдельным заказам из прошлого перебора;
- A. то же семейство «ожидаемый момент следующей покупки» по отдельным позициям;
- B. отдельные потоки заказов: из одной позиции и больших, со скидкой и без, с лояльностью, по основной категории;
- C. структура интервалов между заказами: доля нулевых и коротких интервалов, заказы в последний день,
  давность в долях квантилей интервалов;
- D. число заказов и позиций в непересекающихся интервалах до отсечения и их доли;
- E. календарь: день месяца и недели последнего заказа, заказы в те же календарные месяцы год и два назад;
- каждая группа проверяется по приросту к опорному набору на одних и тех же трёх фолдах.
'''
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines, cust.set_index("customer_id")["win_len"])
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)
data["top_category2"] = data["top_category2"].astype("category")
BASE = [c for c in feats.columns if c != "top_brand"]

cid = "customer_id"
info = cust.set_index(cid)
tr = data[data["part"].eq("train")].reset_index(drop=True)
tr_ids = tr[cid].to_numpy()
y = tr["target"].astype(int).to_numpy()
sl = tr["slice"].to_numpy()
slices = np.unique(sl)
nd = tr["n_days"].to_numpy()
GROUPS = {"2-3": (nd >= 2) & (nd <= 3), "4-5": (nd >= 4) & (nd <= 5), "6-9": (nd >= 6) & (nd <= 9), "10-20": (nd >= 10) & (nd <= 20), "21+": nd >= 21}
print(f"базовых признаков: {len(BASE)} | {(time.time() - T0) / 60:.1f} мин")

## Общие функции

In [ ]:
EST = ["mean", "median", "min", "max", "q25", "q75", "last", "last2", "last3", "ewm", "trim"]
HIST = {"вся": 10_000, "год": 365, "полгода": 182}
LO = pd.Series(float(GAP_DAYS), index=info.index)
HI = (GAP_DAYS + info["win_len"]).astype(float)


def month_idx(s):
    return s.dt.year * 12 + s.dt.month


ref_month = month_idx(info["scoring_date"])


def estimators(E):
    E = E.sort_values([cid, "t"]).reset_index(drop=True)
    g = E.groupby(cid)["t"]
    E["gap"] = g.diff()
    E["rk"] = E.groupby(cid).cumcount()
    gp = E[E["gap"].notna()]
    gg = gp.groupby(cid)["gap"]
    out = pd.DataFrame({"r": g.min(), "n": g.size()})
    out["mean"] = (g.max() - g.min()) / (out["n"] - 1).replace(0, np.nan)
    out["median"] = gg.median()
    out["min"] = gg.min()
    out["max"] = gg.max()
    out["q25"] = gg.quantile(0.25)
    out["q75"] = gg.quantile(0.75)
    out["last"] = gp[gp["rk"] == 1].set_index(cid)["gap"]
    out["last2"] = gp[gp["rk"] <= 2].groupby(cid)["gap"].mean()
    out["last3"] = gp[gp["rk"] <= 3].groupby(cid)["gap"].mean()
    w = 0.7 ** (gp["rk"] - 1)
    out["ewm"] = (gp["gap"] * w).groupby(gp[cid]).sum() / w.groupby(gp[cid]).sum()
    out["trim"] = (gg.sum() - gg.max()) / (gg.count() - 1).replace(0, np.nan)
    return out


def unit_feats(E, prefix, ests=EST, hists=HIST):
    # E: события клиента (customer_id, d), d — дней до отсечения; признаки «ожидаемый момент следующего события»
    out = {}
    E = E.assign(t=E["d"].astype(float))
    for hname, H in hists.items():
        est = estimators(E[E["d"] <= H][[cid, "t"]]).reindex(info.index)
        r = est["r"]
        key = f"{prefix}|{hname}"
        out[f"{key}|n"] = est["n"].fillna(0)
        out[f"{key}|r"] = r
        for e in ests:
            P = est[e].where(est[e] > 0)
            tn, tm = P - r, P - np.mod(r, P)
            pre = f"{key}|{e}"
            out[f"{pre}|t"] = tn
            out[f"{pre}|dist"] = (LO - tn).clip(lower=0) + (tn - HI).clip(lower=0)
            out[f"{pre}|in"] = ((tn > LO) & (tn <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|tm_in"] = ((tm > LO) & (tm <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|k_in"] = np.floor((r + HI) / P) - np.floor((r + LO) / P)
    return out


def block(d):
    return pd.DataFrame(d).reindex(info.index).astype("float32")


def screen(v):
    v = np.asarray(v, dtype=float)
    nan = np.isnan(v)
    row = {"AP@20%": -1.0}
    for sign in (1, -1):
        s = np.where(nan, -np.inf, sign * v)
        ap = float(np.mean([fast_ap(y[sl == q], s[sl == q]) for q in slices]))
        if ap > row["AP@20%"]:
            row = {"AP@20%": ap, "знак": sign}
    row["покрытие"] = 1 - nan.mean()
    aucs = []
    for g, m in GROUPS.items():
        m2 = m & ~nan
        if m2.sum() > 200 and 0 < y[m2].mean() < 1 and np.std(v[m2]) > 0:
            a = roc_auc_score(y[m2], v[m2])
            row[f"AUC {g}"] = max(a, 1 - a)
            if g != "2-3":
                aucs.append(row[f"AUC {g}"])
        else:
            row[f"AUC {g}"] = np.nan
    row["AUC 4+ среднее"] = float(np.mean(aucs)) if aucs else np.nan
    return row

## Заказы

In [ ]:
main = lines.sort_values([cid, "order_id", "paid_price"], ascending=[True, True, False]).drop_duplicates([cid, "order_id"]).set_index("order_id")
o = (
    lines.groupby([cid, "order_id"], sort=False)
    .agg(d=("d", "first"), date=("order_date", "first"), n_items=("paid_price", "size"), order_sum=("paid_price", "sum"),
         disc=("discount_percent", "mean"), loy=("loyalty_percent", "max"))
    .reset_index()
)
o["cat"] = o["order_id"].map(main["category"].astype(str))
o = o.sort_values([cid, "d", "order_id"]).reset_index(drop=True)
print(f"заказов: {len(o):,} | {(time.time() - T0) / 60:.1f} мин")

BLOCKS = {}
BLOCKS["опора: отдельные заказы"] = block(unit_feats(o[[cid, "d"]], "заказ"))
BLOCKS["A: отдельные позиции"] = block(unit_feats(lines[[cid, "d"]], "позиция"))
print(f"A: {(time.time() - T0) / 60:.1f} мин")

STREAMS = {
    "из одной позиции": o["n_items"] == 1,
    "из 2+ позиций": o["n_items"] >= 2,
    "со скидкой": o["disc"] > 0,
    "без скидки": o["disc"] == 0,
    "с лояльностью": o["loy"] > 0,
    "без лояльности": o["loy"] == 0,
    "одежда": o["cat"] == "CLOTHES",
    "обувь": o["cat"] == "FOOTWEAR",
    "прочее": ~o["cat"].isin(["CLOTHES", "FOOTWEAR"]),
}
bfe = {}
for name, m in STREAMS.items():
    bfe.update(unit_feats(o.loc[m, [cid, "d"]], f"поток {name}", ests=["mean", "median", "min", "last", "trim"], hists={"вся": 10_000, "год": 365}))
BLOCKS["B: потоки заказов"] = block(bfe)
print(f"B: {(time.time() - T0) / 60:.1f} мин")

oo = o.copy()
oo["gap"] = oo.groupby(cid)["d"].diff()
gpo = oo[oo["gap"].notna()]
c = {}
for name, cond in [("0", gpo["gap"] == 0), ("le1", gpo["gap"] <= 1), ("le3", gpo["gap"] <= 3), ("le7", gpo["gap"] <= 7), ("le14", gpo["gap"] <= 14),
                   ("le30", gpo["gap"] <= 30), ("gt60", gpo["gap"] > 60), ("gt90", gpo["gap"] > 90), ("gt180", gpo["gap"] > 180)]:
    cf = cond.astype(float)
    c[f"доля интервалов {name}"] = cf.groupby(gpo[cid]).mean()
    c[f"доля интервалов {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).mean()
    c[f"число интервалов {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).sum()
dayc = oo.groupby([cid, "d"]).agg(k=("order_id", "size"), items=("n_items", "sum")).reset_index()
for w in (90, 365):
    c[f"дней с 2+ заказами за {w}"] = ((dayc["k"] >= 2) & (dayc["d"] <= w)).groupby(dayc[cid]).sum()
last_day = dayc.sort_values([cid, "d"]).drop_duplicates(cid).set_index(cid)
c["заказов в последний день"] = last_day["k"]
c["позиций в последний день"] = last_day["items"]
c["давность последнего дня с 2+ заказами"] = dayc[dayc["k"] >= 2].groupby(cid)["d"].min()
r_o = oo.groupby(cid)["d"].min()
qs = gpo.groupby(cid)["gap"].quantile([0.25, 0.5, 0.75, 0.9]).unstack()
for q in qs.columns:
    c[f"давность / квантиль {q}"] = r_o / qs[q].replace(0, np.nan)
    c[f"давность - квантиль {q}"] = r_o - qs[q]
c["разброс интервалов между заказами"] = gpo.groupby(cid)["gap"].std() / gpo.groupby(cid)["gap"].mean().replace(0, np.nan)
pos = gpo[gpo["gap"] > 0].groupby(cid)["gap"]
c["средний ненулевой интервал"] = pos.mean()
c["давность - средний ненулевой интервал"] = r_o - pos.mean()
BLOCKS["C: структура интервалов"] = block(c)

BINS = [0, 7, 14, 30, 60, 90, 120, 180, 270, 365, 450, 540, 733]
labels = [f"{BINS[i]}-{BINS[i + 1]}" for i in range(len(BINS) - 1)]
dcnt = {}
ob = pd.cut(oo["d"], BINS, right=False, labels=labels)
lb = pd.cut(lines["d"], BINS, right=False, labels=labels)
t_o = pd.crosstab(oo[cid], ob).reindex(columns=labels, fill_value=0)
t_l = pd.crosstab(lines[cid], lb).reindex(columns=labels, fill_value=0)
for lab in labels:
    dcnt[f"заказов {lab}"] = t_o[lab]
    dcnt[f"позиций {lab}"] = t_l[lab]
    dcnt[f"доля заказов {lab}"] = t_o[lab] / t_o.sum(axis=1)
BLOCKS["D: заказы по интервалам"] = block(dcnt)

mi = month_idx(oo["date"])
rel = oo[cid].map(ref_month) - mi          # месяцев до месяца скоринга
e = {}
last_o = oo.drop_duplicates(cid).set_index(cid)
e["день месяца последнего заказа"] = last_o["date"].dt.day
e["день недели последнего заказа"] = last_o["date"].dt.dayofweek
e["месяц последнего заказа"] = last_o["date"].dt.month
e["доля заказов 1–10 числа"] = (oo["date"].dt.day <= 10).groupby(oo[cid]).mean()
e["доля заказов 21+ числа"] = (oo["date"].dt.day >= 21).groupby(oo[cid]).mean()
e["заказов в месяцы окна год назад"] = rel.between(10, 12).groupby(oo[cid]).sum()
e["заказов в месяцы окна два года назад"] = rel.between(22, 24).groupby(oo[cid]).sum()
e["заказов в 3 месяца перед окном год назад"] = rel.between(13, 15).groupby(oo[cid]).sum()
e["заказов в 3 месяца после окна год назад"] = rel.between(7, 9).groupby(oo[cid]).sum()
e["активных месяцев"] = mi.groupby(oo[cid]).nunique()
e["активных месяцев за год"] = mi.where(rel <= 12).groupby(oo[cid]).nunique()
BLOCKS["E: календарь"] = block(e)
print({k: v.shape[1] for k, v in BLOCKS.items()}, f"| {(time.time() - T0) / 60:.1f} мин")

## Оценка по одиночке

In [ ]:
rows = {}
for bname, B in BLOCKS.items():
    Bt = B.reindex(tr_ids).reset_index(drop=True)
    for col in B.columns:
        rows[(bname, col)] = screen(Bt[col].to_numpy())
for col in ["formula", "days_formula", "tnext_mean_dist"]:
    rows[("база", col)] = screen(tr[col].to_numpy())
Sc = pd.DataFrame.from_dict(rows, orient="index")
Sc.index = pd.MultiIndex.from_tuples(Sc.index, names=["группа", "признак"])
print(f"оценка: {(time.time() - T0) / 60:.1f} мин")
print("Лучшие по средней AUC внутри групп с 4+ днями заказов:")
display(Sc.sort_values("AUC 4+ среднее", ascending=False).head(40))
print("Лучшие по AP@20% в одиночку:")
display(Sc.sort_values("AP@20%", ascending=False).head(20))
print("Лучший признак каждой группы:")
display(Sc.groupby(level=0)[["AP@20%", "AUC 2-3", "AUC 4-5", "AUC 6-9", "AUC 10-20", "AUC 21+", "AUC 4+ среднее"]].max())

## Прирост групп на LightGBM

In [ ]:
PARAMS = dict(objective="binary", learning_rate=0.05, n_estimators=3000, num_leaves=31, min_child_samples=100, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.5, reg_lambda=5.0, cat_smooth=50, random_state=SEED, n_jobs=-1, verbose=-1)
folds = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(tr, tr["slice"] + "_" + tr["target"].astype(int).astype(str)))

frame = tr.copy()
NAMES, COLS = {}, {}
for i, (bname, B) in enumerate(BLOCKS.items()):
    safe = [f"b{i}_{j}" for j in range(B.shape[1])]
    frame[safe] = B.reindex(tr_ids).to_numpy()
    COLS[bname] = safe
    NAMES.update(dict(zip(safe, [f"{bname} :: {c}" for c in B.columns])))


def run_lgb(features):
    p, imp = np.zeros(len(frame)), pd.Series(0.0, index=features)
    for a, b in folds:
        m = lgb.LGBMClassifier(**PARAMS).fit(frame[features].iloc[a], y[a], eval_set=[(frame[features].iloc[b], y[b])], callbacks=[lgb.early_stopping(100, verbose=False)])
        p[b] = m.predict_proba(frame[features].iloc[b])[:, 1]
        g = pd.Series(m.booster_.feature_importance("gain"), index=features)
        imp += g / g.sum() / len(folds)
    cells = np.array([[fast_ap(y[b][sl[b] == q], p[b][sl[b] == q]) for q in slices] for _, b in folds])
    return cells, imp


RES = {}
RES["222 признака"] = run_lgb(BASE)
REF = BASE + COLS["опора: отдельные заказы"]
RES["опора: 222 + отдельные заказы"] = run_lgb(REF)
for bname in [k for k in BLOCKS if k != "опора: отдельные заказы"]:
    RES[f"опора + {bname}"] = run_lgb(REF + COLS[bname])
    print(f"{bname}: {RES[f'опора + {bname}'][0].mean():.4f} | {(time.time() - T0) / 60:.1f} мин")
ALL_NEW = REF + [c for k, v in COLS.items() if k != "опора: отдельные заказы" for c in v]
RES["опора + все новые группы"] = run_lgb(ALL_NEW)

tab = {}
for name, (cells, _) in RES.items():
    base_cells = RES["222 признака"][0] if name == "опора: 222 + отдельные заказы" else RES["опора: 222 + отдельные заказы"][0]
    d = (cells - base_cells).ravel()
    tab[name] = {"AP@20%": cells.mean(), "прирост": d.mean(), "разброс": d.std(), "плюс в парах": f"{int((d > 0).sum())} из {len(d)}"}
tab["222 признака"].update({"прирост": np.nan, "разброс": np.nan, "плюс в парах": ""})
print("Прирост: для опоры — к 222 признакам, для групп — к опоре")
display(pd.DataFrame(tab).T)

imp = RES["опора + все новые группы"][1]
imp_new = imp.drop(BASE)
print("Доля важности по группам в наборе «опора + все новые»:")
grp = {c: k for k, v in COLS.items() for c in v}
by_group = imp_new.groupby(imp_new.index.map(grp)).sum()
by_group["222 признака"] = imp[BASE].sum()
display(by_group.sort_values(ascending=False).to_frame("доля").T)
print("Самые важные новые признаки:")
display(imp_new.sort_values(ascending=False).head(30).rename(index=NAMES).to_frame("доля важности"))
print(f"Всего времени: {(time.time() - T0) / 60:.1f} мин")